# Flight Delay Time Statistics Dashboard: My Personal Exploration

In this notebook, I build and analyze a dashboard for flight delay statistics. My goal is to understand the patterns behind flight delays and create visualizations that help me (and others) gain insights from the data.

```markdown
## My Approach

I designed the dashboard layout, wrote the callback logic, and used Plotly Express for the visualizations. Each graph updates based on the year you enter, making it easy to spot trends and outliers. Below is a high-level look at my process, which I refined through trial and error until I was happy with the results.

```python
app.layout = html.Div(children=[ 
  
                                html.H1('...TODO1...', style={...TODO1...}),
                                
                                html.Div(["Input Year: ", dcc.Input(id='...TODO2...', value='...TODO2...', type='...TODO2...', style={'...TODO2...'}),], style={'font-size': 30}),
                                html.Br(),
                                html.Br(),
                          
                                html.Div([
                                        html.Div('...TODO3...'),
                                        html.Div('...TODO3...')
                                ], style={'display': 'flex'}),
    
                                html.Div([
                                        html.Div('...TODO3...'),
                                        html.Div('...TODO3...')
                                ], style={'display': 'flex'}),
                                
                                html.Div('...TODO3...', style={'width':'65%'})
                       
                                ])

# add callback decorator
@app.callback( [
               Output(...TODO4...),
               Output(...TODO4...),
               Output(...TODO4...),
               Output(...TODO4...),
               Output(...TODO4...)
               ],
               Input(...TODO4...))
               
# Add computation to callback function and return graph
def get_graph(entered_year):
    
    # Compute required information for creating graph from the data
    ...TODO5..., ...TODO5..., ...TODO5..., ...TODO5..., ...TODO5... = compute_info(airline_data, entered_year)
            
    # Create graph
    carrier_fig = px.line(...TODO6..., x='...TOTO6...', y='...TOTO6...', color='...TOTO6...', title='...TOTO6...')
    weather_fig = px.line(...TODO6..., x='...TOTO6...', y='...TOTO6...', color='...TOTO6...', title='...TOTO6...')
    nas_fig = px.line(...TODO6..., x='...TOTO6...', y='...TOTO6...', color='...TOTO6...', title='...TOTO6...')
    sec_fig = px.line(...TODO6..., x='...TOTO6...', y='...TOTO6...', color='...TOTO6...', title='...TOTO6...')
    late_fig = px.line(...TODO6..., x='...TOTO6...', y='...TOTO6...', color='...TOTO6...', title='...TOTO6...')
            
    return[carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]


# Run the app
if __name__ == '__main__':
    app.run_server(mode='jupyterlab')
```
````


## Why Analyze Flight Delays?

I'm interested in what causes flight delays and how they vary across airlines, airports, and time. This dashboard is my way of exploring those questions using data and visualization.

## How I Built It

- I started by designing a clean, simple layout with Dash and HTML components.
- I created input fields and graph placeholders for each delay type.
- I wrote a callback function to update all graphs when the year changes.
- I used pandas to group and average the data, and Plotly Express to make the line charts.
- I tweaked the style and layout until it felt intuitive and visually appealing.


In [ ]:
# Author: Mohammad Sayem Chowdhury
# Importing the libraries I use for dashboarding and data analysis
import pandas as pd
import dash
from dash import html 
from dash import dcc
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash
import plotly.express as px

# Creating my Dash application
app = JupyterDash(__name__)
JupyterDash.infer_jupyter_proxy_config()

# Load the airline data into a pandas DataFrame
# I use this real-world dataset to explore delay patterns
airline_data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
                          encoding = "ISO-8859-1",
                          dtype={'Div1Airport': str, 'Div1TailNum': str, 
                                 'Div2Airport': str, 'Div2TailNum': str})

# My function to compute average delays for each type
# This helps me break down the data for each graph

def compute_info(airline_data, entered_year):
    # Filter data for the selected year
    df = airline_data[airline_data['Year'] == int(entered_year)]
    # Calculate monthly averages for each delay type
    avg_car = df.groupby(['Month','Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month','Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month','Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month','Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month','Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late

# Building the dashboard layout
# I keep it clean and focused on the key delay types
app.layout = html.Div(children=[
    html.H1('Flight Delay Time Statistics',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                        type='number', style={'height':'35px', 'font-size': 30})],
             style={'font-size': 30}),
    html.Br(),
    html.Br(),
    html.Div([
        html.Div(dcc.Graph(id='carrier-plot')),
        html.Div(dcc.Graph(id='weather-plot'))
    ], style={'display': 'flex'}),
    html.Div([
        html.Div(dcc.Graph(id='nas-plot')),
        html.Div(dcc.Graph(id='security-plot'))
    ], style={'display': 'flex'}),
    html.Div(dcc.Graph(id='late-plot'), style={'width':'65%'})
])

# My callback function updates all graphs when the year changes
# This makes the dashboard interactive and responsive
@app.callback([
    Output(component_id='carrier-plot', component_property='figure'),
    Output(component_id='weather-plot', component_property='figure'),
    Output(component_id='nas-plot', component_property='figure'),
    Output(component_id='security-plot', component_property='figure'),
    Output(component_id='late-plot', component_property='figure')
],
    Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # Compute the data for each graph
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)
    # Create the line charts for each delay type
    carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline', title='Average carrier delay time (minutes) by airline')
    weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline', title='Average weather delay time (minutes) by airline')
    nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline', title='Average NAS delay time (minutes) by airline')
    sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline', title='Average security delay time (minutes) by airline')
    late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline', title='Average late aircraft delay time (minutes) by airline')
    return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

# To run the app in Jupyter, uncomment the line below
# app.run_server(mode="inline", host="localhost", port=7645, debug=True)


## My Approach

I experiment with different ways to visualize and analyze flight delay data, documenting my process and the insights I find most interesting.

---

Working on this dashboard was a great way to learn Dash and interactive data visualization. I encourage you to experiment with the layout, try new features, and make the dashboard your own!

*Notebook by Mohammad Sayem Chowdhury*


## Reflections

This section is for my thoughts on building the dashboard, including what I found useful and what I want to explore further.

Building this dashboard taught me a lot about interactive data visualization and the power of Python for real-world analytics. I enjoyed experimenting with different layouts and seeing how small changes in the data can reveal big trends in airline performance. If you have ideas for new features or want to collaborate, let me know!

**Notebook by Mohammad Sayem Chowdhury**
